# EXP11: SL/TP 非对称比率扫描

固定 Trail 参数，扫 SL x TP 矩阵找最优风险收益比。

SL [3.0, 3.5, 4.0, 4.5, 5.0, 5.5] x TP [3.0, 4.0, 5.0, 6.0] = 24 变体

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Part 1: SL x TP 网格

In [ ]:
results = []
for sl in [3.0, 3.5, 4.0, 4.5, 5.0, 5.5]:
    for tp in [3.0, 4.0, 5.0, 6.0]:
        r = run_variant(data, f"SL{sl}_TP{tp}",
            **{**LIVE_KWARGS, "sl_atr_mult": sl, "tp_atr_mult": tp})
        r['sl'] = sl
        r['tp'] = tp
        r['ratio'] = round(tp / sl, 2)
        results.append(r)

print_ranked(results)

In [ ]:
df = pd.DataFrame([{'sl': r['sl'], 'tp': r['tp'], 'ratio': r['ratio'],
                     'sharpe': r['sharpe'], 'total_pnl': r['total_pnl'],
                     'max_dd': r.get('max_dd', 0), 'n': r['n']} for r in results])

print("\nSharpe Heatmap (rows=SL, cols=TP):")
pivot_s = df.pivot_table(index='sl', columns='tp', values='sharpe')
print(pivot_s.round(2))

print("\nPnL Heatmap:")
pivot_p = df.pivot_table(index='sl', columns='tp', values='total_pnl')
print(pivot_p.round(0))

print("\nMaxDD Heatmap:")
pivot_d = df.pivot_table(index='sl', columns='tp', values='max_dd')
print(pivot_d.round(0))

In [ ]:
# 按 TP/SL 比率分组
print("\n=== By TP/SL Ratio ===")
print(df.groupby('ratio')[['sharpe', 'total_pnl', 'n']].mean().round(2))

## Part 2: 冠军 K-Fold

In [ ]:
best = sorted(results, key=lambda r: r['sharpe'], reverse=True)[0]
print(f"Champion: SL={best['sl']}, TP={best['tp']}, Ratio={best['ratio']}, Sharpe={best['sharpe']:.2f}")

champ_kwargs = {**LIVE_KWARGS, "sl_atr_mult": best['sl'], "tp_atr_mult": best['tp']}
champ_folds = run_kfold(data, champ_kwargs, label_prefix="Champ_")
base_folds = run_kfold(data, LIVE_KWARGS, label_prefix="Base_")

print("\n=== K-Fold ===")
for b, c in zip(base_folds, champ_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  Champ={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")

In [ ]:
# TP 触发率 vs SL 触发率
print("\n=== Exit Breakdown ===")
for r in results:
    trades = r['_trades']
    n = len(trades)
    tp_n = sum(1 for t in trades if t.exit_reason == 'TP')
    sl_n = sum(1 for t in trades if t.exit_reason == 'SL')
    trail_n = sum(1 for t in trades if t.exit_reason == 'Trailing')
    print(f"SL{r['sl']}/TP{r['tp']}: TP={tp_n}({tp_n/n*100:.0f}%) SL={sl_n}({sl_n/n*100:.0f}%) "
          f"Trail={trail_n}({trail_n/n*100:.0f}%) Sharpe={r['sharpe']:.2f}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp11_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved')